# Index Health: Fresh, Valid, and Batched

Level 3: RAG and Retrieval Engineering.

**Problems in this notebook**

- Problem 23: Detect Stale Embeddings
- Problem 24: Validate an Embedding Record
- Problem 26: Batch Documents for Embedding

**How to use this notebook**

1. Read the problem and say your plan aloud before you type anything.
2. Paste the AI prompt given (or write your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference only after you have tried.

The ideas behind these problems are in `02_index_health_THEORY.md` in this folder.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 23: Detect Stale Embeddings

**The situation**

A policy document was edited last week, but its embeddings were made last month. The assistant still answers from the old text, and the customer sees an out-of-date answer.

**Your task**

Write `find_stale_docs(docs)` that returns the IDs of documents that need to be embedded again.

**Example**

`[{"doc_id": "d1", "updated_at": 10, "embedded_at": 5}]` -> `["d1"]`

**Ask before coding**

- Are both timestamps in the same format and timezone?
- If a timestamp is missing, should we re-embed to be safe?
- Do you want whole documents back, or individual chunks?

**Plan to say aloud**

1. Loop over the documents in order.
2. Skip any row with no `doc_id`, since we cannot act on it.
3. Mark it stale if either timestamp is missing.
4. Otherwise mark it stale if `updated_at` is later than `embedded_at`.
5. Return the stale IDs in input order.

### Prompt for your AI assistant

```text
Write a Python function find_stale_docs(docs: list[dict]) -> list[str].
Each doc may have "doc_id", "updated_at" and "embedded_at" (numbers).
Rules:
- skip docs with no doc_id
- a doc is stale if updated_at or embedded_at is missing or None
- a doc is stale if updated_at > embedded_at (equal is fresh)
- return stale doc IDs in input order
Show only the function.
```

**Check the AI's code for**

- Uses `doc.get("updated_at", 0)`, which quietly treats a missing update time as fresh.
- Writes `>=` instead of `>`, so a document embedded at the same moment it was saved gets re-embedded forever.
- Tests `if not embedded_at`, which treats a real timestamp of 0 as missing.
- Crashes with `KeyError` on a row with no `doc_id` instead of skipping it.

In [ ]:
def find_stale_docs(docs: list[dict]) -> list[str]:
    """Return IDs of docs whose embeddings are older than the source.

    Rules:
    - skip docs with no doc_id
    - stale if updated_at or embedded_at is missing/None (re-embed to be safe)
    - stale if updated_at > embedded_at; equal timestamps are fresh
    - keep input order
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([{"doc_id": "d1", "updated_at": 10, "embedded_at": 5}],), ["d1"]),
    (([{"doc_id": "d1", "updated_at": 5, "embedded_at": 10}],), []),
    # equal timestamps are fresh
    (([{"doc_id": "d1", "updated_at": 7, "embedded_at": 7}],), []),
    # missing timestamps are stale; missing doc_id is skipped
    (([{"doc_id": "a", "updated_at": 3},
       {"doc_id": "b", "embedded_at": 3},
       {"updated_at": 9, "embedded_at": 1},
       {"doc_id": "c", "updated_at": 1, "embedded_at": 2}],), ["a", "b"]),
    # a timestamp of 0 is a real value, not "missing"
    (([{"doc_id": "z", "updated_at": 0, "embedded_at": 0}],), []),
    (([],), []),
]
run_checks(find_stale_docs, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def find_stale_docs_reference(docs: list[dict]) -> list[str]:
    stale = []
    for doc in docs:
        doc_id = doc.get("doc_id")
        if not doc_id:
            continue
        updated, embedded = doc.get("updated_at"), doc.get("embedded_at")
        if updated is None or embedded is None or updated > embedded:
            stale.append(doc_id)
    return stale


assert run_checks(find_stale_docs_reference, cases)

**Explain it in 30 seconds**

> "A document is stale when it changed after we embedded it, so I compare the two timestamps. If either timestamp is missing I flag it anyway, because re-embedding costs a little and a wrong answer costs trust. Rows with no ID are skipped since we cannot act on them. Retrieval can work perfectly and still return old text if the index is stale."

**Edge cases to mention**

- Timestamps in different timezones or formats, so the comparison is wrong.
- Clock skew between the source system and the indexer.
- A document deleted at the source but still in the index.
- A document where only some chunks were re-embedded.

**If they ask "how would this work in production?"**

- Store a content hash and compare it, so an edit with no real change does not trigger work.
- Store the embedding model version too, and re-embed everything when the model changes.
- Put stale IDs on a re-embedding queue and show a freshness number per connector and per tenant.

## Problem 24: Validate an Embedding Record

**The situation**

Before a chunk goes into the vector index, its record must carry who owns it and who can see it. A record with no tenant or no roles could leak to the wrong customer, and a vector of the wrong size breaks search.

**Your task**

Write `validate_embedding_record(record, expected_dim)` that returns a list of problems. An empty list means the record is safe to insert.

**Example**

A full record with a 3-number vector and `expected_dim=3` -> `[]`. Same record with no `tenant_id` -> `["missing tenant_id"]`

**Ask before coding**

- Which fields are required?
- Is an empty roles list allowed, or does every record need at least one role?
- Should we reject the record, or insert it with a warning?

**Plan to say aloud**

1. Check each required text field is present and not empty: tenant, doc ID, source, model name.
2. Check roles is a non-empty list. Missing roles must never mean "everyone".
3. Check the vector is a list of numbers with exactly the expected length.
4. Collect every problem, not just the first, and return the list.

### Prompt for your AI assistant

```text
Write a Python function validate_embedding_record(record: dict, expected_dim: int) -> list[str].
Return a list of error strings, in this order:
- "missing <field>" for each of tenant_id, doc_id, source, embedding_model that is missing or empty
- "missing roles" if roles is not a non-empty list
- "vector must be <expected_dim> numbers" if vector is not a list of int/float of exactly that length
Collect all errors, do not stop at the first. Return [] when valid. Show only the function.
```

**Check the AI's code for**

- Returns on the first error, so the ingestion log only ever shows one problem per record.
- Forgets to check roles at all, so a record with no access rules goes in.
- Checks the vector length but not that the items are numbers.
- Raises an exception instead of returning the list the spec asks for.

In [ ]:
def validate_embedding_record(record: dict, expected_dim: int) -> list[str]:
    """Return a list of problems with an embedding record ([] means valid).

    Errors, in this order:
    - "missing <field>" for tenant_id, doc_id, source, embedding_model (missing or empty)
    - "missing roles" unless roles is a non-empty list
    - "vector must be <expected_dim> numbers" unless vector is a list of numbers of that length
    """
    raise NotImplementedError

In [ ]:
GOOD = {"tenant_id": "acme", "doc_id": "d1", "source": "drive", "embedding_model": "embed-v2",
        "roles": ["hr"], "vector": [0.1, 0.2, 0.3]}
cases = [
    ((GOOD, 3), []),
    (({**GOOD, "tenant_id": None}, 3), ["missing tenant_id"]),
    (({**GOOD, "vector": [0.1, 0.2]}, 3), ["vector must be 3 numbers"]),
    (({**GOOD, "roles": []}, 3), ["missing roles"]),
    (({**GOOD, "vector": [0.1, "x", 0.3]}, 3), ["vector must be 3 numbers"]),
    (({}, 2), ["missing tenant_id", "missing doc_id", "missing source", "missing embedding_model",
               "missing roles", "vector must be 2 numbers"]),
]
run_checks(validate_embedding_record, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def validate_embedding_record_reference(record: dict, expected_dim: int) -> list[str]:
    errors = []
    for field in ("tenant_id", "doc_id", "source", "embedding_model"):
        if not record.get(field):
            errors.append(f"missing {field}")
    roles = record.get("roles")
    if not isinstance(roles, list) or not roles:
        errors.append("missing roles")
    vector = record.get("vector")
    if (not isinstance(vector, list) or len(vector) != expected_dim
            or not all(isinstance(x, (int, float)) for x in vector)):
        errors.append(f"vector must be {expected_dim} numbers")
    return errors


assert run_checks(validate_embedding_record_reference, cases)

**Explain it in 30 seconds**

> "I check the record before it reaches the index. Tenant, document ID, source, and model name must be filled in, roles must be a real non-empty list, and the vector must have the exact size the model produces. I collect every problem so one log line tells the whole story. Metadata in RAG is not decoration: it decides who can see the chunk."

**Edge cases to mention**

- Missing tenant ID, which risks showing one customer's data to another.
- Missing or empty roles, which must fail closed, never default to public.
- A vector from a different model, so its length is wrong.
- A vector that contains a string or None from a bad upstream job.

**If they ask "how would this work in production?"**

- Enforce the same schema in the ingestion service and reject bad records before the index sees them.
- Also store a content hash, index version, and connector checkpoint for debugging and re-sync.
- Count rejected records per connector and alert when the rate jumps.

## Problem 26: Batch Documents for Embedding

**The situation**

A customer uploads thousands of documents at once. The embedding API accepts only a limited number of items per call, so the ingestion job must split the work into batches.

**Your task**

Write `make_batches(items, batch_size)` that splits a list into batches of at most `batch_size`, keeping order.

**Example**

`[1, 2, 3, 4, 5]`, `batch_size=2` -> `[[1, 2], [3, 4], [5]]`

**Ask before coding**

- Is the API limit on item count, or on tokens per request?
- If one batch fails, do we retry just that batch?
- Must the original order be kept, for example to match results back to IDs?

**Plan to say aloud**

1. Reject a batch size of 0 or less with a clear error.
2. Walk the list in steps of `batch_size`.
3. Take a slice at each step. The last slice may be shorter.
4. Return the list of slices.

### Prompt for your AI assistant

```text
Write a Python function make_batches(items: list, batch_size: int) -> list[list].
Rules:
- raise ValueError if batch_size <= 0
- return consecutive batches of at most batch_size items, keeping order
- the last batch may be smaller; an empty input returns []
Use list slicing, no third-party libraries. Show only the function.
```

**Check the AI's code for**

- Loops forever or crashes with a confusing error when `batch_size` is 0.
- Drops the last short batch, so the final few documents are never embedded.
- Returns `[[]]` for an empty input, which would send an empty request to the API.
- Uses a generator when the checks expect a list.

In [ ]:
def make_batches(items: list, batch_size: int) -> list[list]:
    """Split items into consecutive batches of at most batch_size.

    Rules:
    - ValueError if batch_size <= 0
    - keep order; the last batch may be smaller
    - empty input returns []
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([1, 2, 3, 4, 5], 2), [[1, 2], [3, 4], [5]]),
    (([1, 2, 3], 1), [[1], [2], [3]]),
    ((["a", "b"], 10), [["a", "b"]]),
    (([], 3), []),
]
run_checks(make_batches, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def make_batches_reference(items: list, batch_size: int) -> list[list]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")
    return [items[i:i + batch_size] for i in range(0, len(items), batch_size)]


assert run_checks(make_batches_reference, cases)

try:
    make_batches_reference([1, 2], 0)
    raise AssertionError("expected ValueError")
except ValueError:
    print("batch_size 0 rejected")

**Explain it in 30 seconds**

> "I step through the list by the batch size and take a slice each time, so order is kept and the last batch can be short. A batch size of zero or less is a bug, so I raise a clear error. Fixed-size batches are the starting point; real ingestion also limits by tokens and saves progress so a failed batch can be retried alone."

**Edge cases to mention**

- A batch size of zero or a negative number.
- An empty input list.
- A batch size bigger than the whole list.
- Items that are small in count but huge in tokens, which still break the API limit.

**If they ask "how would this work in production?"**

- Batch by token count and payload size, not just item count.
- Save a checkpoint after each batch and retry failed batches without redoing the good ones.
- Track throughput and failure rate per ingestion job.

## Wrap-up

- These problems guard the index itself: what goes in, when it gets refreshed, and how it gets loaded.
- When data is missing, choose the safe side: re-embed, reject the record, or raise a clear error.
- Good answers depend on a healthy index, so ingestion checks are part of answer quality.